# SIMBA-25 analogues of CEERS nirspec7-000719 / nirspec7-002962

Same goal as `analogues_specphot_cis100.ipynb`, but drawing analogues from the **cis25 quenched
selection** (`powderday_flux_quenched_m25`, z = 1.0 anchor: snap 105, z = 0.993, 44 galaxies).
Unlike the cis100 PSB sample, this anchor contains genuinely **old** quenched galaxies
(mass-weighted ages up to 3.7 Gyr) and masses up to log M* = 11.45, so both targets can be
matched at fixed mass AND age with tight windows.

Trade-offs vs cis100 (all handled/flagged below):
- **RT not yet run for snap 105** — everything is staged
  (`sed_quenched_regions/dusty_simdust/powderday_sed_out/snap_105/master.snap105.job`,
  per-galaxy configs, particle files); submit that job and the SED cells below fill in.
  The selection/sanity/dust-mass parts work already.
- **No AGN-host analogues exist in the 25 Mpc box** at this anchor (only 3/44 galaxies have
  f_Edd ≥ 0.02, all young/low-mass) — the AGN/non-AGN split collapses to non-AGN only.
  For 002962 compare against non-AGN hosts and the AGN-subtracted CIGALE curve.
- Single anchor epoch: all analogues sit at z = 0.993 (Δt ≈ 0.6–1.1 Gyr after the targets);
  they are placed at each target's redshift for flux normalization.
- m25 SEDs have 4 sightlines × 5 apertures (1/3.2/10/31.6/100 kpc); each sightline enters the
  stack as its own curve, aperture is configurable (default: largest = total 100 kpc region).


In [ ]:
# ── Part 0 · config ──────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck13          # same cosmology as simbanator.sed.makesed
import astropy.units as u
from astropy import constants as const

plt.rcParams.update({'font.size': 13})

HOME     = '/mnt/home/glorenzon/analize_simba_cgm'
CIS_OUT  = os.path.join(HOME, 'output', 'cis25')
SED_ROOT = os.path.join(CIS_OUT, 'sed_quenched_regions', 'dusty_simdust', 'powderday_sed_out')
SEL_FITS = os.path.join(CIS_OUT, 'tables', 'powderday_quenched_selection.fits')
HIST_H5  = os.path.join(CIS_OUT, 'caesar_sfh', 'history_anchor_z1.hdf5')
BH_H5    = os.path.join(CIS_OUT, 'caesar_sfh', 'bh_history_anchor_z1.hdf5')
OBS_DIR  = os.path.join(HOME, 'obs_data', 'pilot_specphot_cigale')   # staged CIGALE fiducial outputs
OUTDIR   = os.path.join(CIS_OUT, 'analogues_specphot')
os.makedirs(OUTDIR, exist_ok=True)

ANCHOR = dict(z_target=1.0, snap=105, z_snap=0.993)

# fixed entity colors (Okabe-Ito CVD-safe pair); observed data is always black
COL = {'AGN': '#E69F00', 'no_AGN': '#0072B2', 'obs': 'k'}

# ── observed targets (fiducial: cigale_ppxfprior_agn + final_consensus.md) ──
# Windows are TIGHT here (unlike cis100): the anchor has properly matched galaxies.
# Counts at these defaults: 719 -> 15 analogues, 2962 -> 8.
TARGETS = {
    '719': dict(
        z=1.463, logm=10.54, logm_err=0.05, age_gyr=2.35, agn=False,
        logmdust=6.74, logmdust_err=0.6,          # err > value -> treat as ~upper limit
        best_model='719_best_model.fits', obs_id='719',
        dlogm=0.30, dage_gyr=0.75,
    ),
    '2962': dict(
        z=1.266, logm=10.92, logm_err=0.05, age_gyr=3.30, agn=True,
        logmdust=7.17, logmdust_err=0.5,
        best_model='2962_best_model.fits', obs_id='2962',
        dlogm=0.40, dage_gyr=1.0,                 # slightly wider: only 4 tight matches exist
    ),
}

# AGN host = radiatively efficient accretor (same thresholds as the cis100 notebook)
EPS_RAD      = 0.1
AGN_LBOL_MIN = 1e43        # erg/s
AGN_FEDD_MIN = 0.02

# m25 SED layout
APERTURE_IDX = -1          # Hyperion aperture index; -1 = largest (total 100 pkpc region)
print('output ->', OUTDIR)


In [ ]:
# ── Part 1 · observed CIGALE SEDs + photometry (identical to cis100 notebook) ─
AGN_COLS = ('agn.SKIRTOR2016_torus', 'agn.SKIRTOR2016_polar_dust', 'agn.SKIRTOR2016_disk')

def load_best_model(fname):
    with fits.open(os.path.join(OBS_DIR, fname)) as f:
        d = f[1].data
        wl_um  = np.asarray(d['wavelength'], float) * 1e-3        # nm -> micron (observed frame)
        fnu    = np.asarray(d['Fnu'], float)                      # mJy
        ltot   = np.asarray(d['L_lambda_total'], float)
        lagn   = np.zeros_like(ltot)
        for c in AGN_COLS:
            if c in d.columns.names:
                lagn += np.asarray(d[c], float)
        with np.errstate(invalid='ignore', divide='ignore'):
            fgal = fnu * np.clip(1.0 - lagn / ltot, 0.0, 1.0)
    s = np.argsort(wl_um)
    return wl_um[s], fnu[s], fgal[s]

OBS_BANDS = {
    'cfht.megacam.u': 0.381, 'cfht.megacam.g': 0.487, 'cfht.megacam.r': 0.625,
    'cfht.megacam.i': 0.770, 'cfht.megacam.z': 0.888,
    'hst.acs.wfc.F606W': 0.592, 'hst.acs.wfc.F814W': 0.805,
    'hst.wfc3.ir.F125W': 1.248, 'hst.wfc3.ir.F160W': 1.537,
    'cfht.wircam.J': 1.253, 'cfht.wircam.H': 1.631, 'cfht.wircam.Ks': 2.146,
    'spitzer.irac.I1': 3.55, 'spitzer.irac.I2': 4.49, 'spitzer.irac.I3': 5.73, 'spitzer.irac.I4': 7.87,
    'jwst.miri.F770W': 7.70, 'jwst.miri.F1000W': 10.0, 'jwst.miri.F1280W': 12.8,
    'jwst.miri.F1500W': 15.0, 'jwst.miri.F1800W': 18.0, 'jwst.miri.F2100W': 21.0,
}

def load_observations(obs_id):
    with fits.open(os.path.join(OBS_DIR, 'observations.fits')) as f:
        d = f[1].data
        row = d[np.asarray([str(i).strip() for i in d['id']]) == obs_id][0]
    wl, fl, er = [], [], []
    for band, piv in OBS_BANDS.items():
        if band not in d.columns.names:
            continue
        v = float(row[band]); e = float(row[band + '_err'])
        if np.isfinite(v) and v > 0:
            wl.append(piv); fl.append(v); er.append(e)
    s = np.argsort(wl)
    return np.asarray(wl)[s], np.asarray(fl)[s], np.asarray(er)[s]

OBS = {}
for tid, T in TARGETS.items():
    wl_m, f_tot, f_gal = load_best_model(T['best_model'])
    wl_p, f_p, e_p = load_observations(T['obs_id'])
    OBS[tid] = dict(model_wl=wl_m, model_tot=f_tot, model_gal=f_gal, phot_wl=wl_p, phot_f=f_p, phot_e=e_p)
    print(f"{tid}: best-model {wl_m.size} pts, photometry {wl_p.size} bands "
          f"({wl_p.min():.2f}-{wl_p.max():.1f} um)")


In [ ]:
# ── Part 2 · cis25 anchor pool: selection table + history + BH at snap 105 ───
SEL = Table.read(SEL_FITS)
POOL = SEL[SEL['z_target'] == ANCHOR['z_target']]

with h5py.File(HIST_H5, 'r') as f:
    _gids = np.asarray(f['metadata/galaxy_ids'][:], int)
    _age  = np.asarray(f['properties/ages.mass_weighted'][0, :], float)   # row 0 = anchor snap
    _ms   = np.asarray(f['properties/masses.stellar'][0, :], float)
    _md   = np.asarray(f['properties/masses.dust'][0, :], float)
with h5py.File(BH_H5, 'r') as f:
    _bhm, _bmd, _bfe = (np.asarray(f[k][0, :], float) for k in ('bh_mass', 'bh_mdot', 'bh_fedd'))

_idx = {g: i for i, g in enumerate(_gids)}
cols = np.array([_idx[int(g)] for g in POOL['gal_id']])
POOL['AGE_STAR']  = _age[cols]
POOL['LOG_MSTAR'] = np.log10(_ms[cols])          # history-based (POOL['log_mstar'] is the same source)
with np.errstate(invalid='ignore'):
    POOL['LOG_MDUST'] = np.log10(np.where(_md[cols] > 0, _md[cols], np.nan))
POOL['BH_MASS'] = _bhm[cols]
POOL['FEDD']    = _bfe[cols]
POOL['LBOL']    = (EPS_RAD * (_bmd[cols] * u.Msun / u.yr) * const.c**2).to(u.erg / u.s).value
POOL['IS_AGN']  = (np.nan_to_num(POOL['FEDD']) >= AGN_FEDD_MIN) & \
                  (np.nan_to_num(POOL['LBOL']) >= AGN_LBOL_MIN)

def rtout_path(snap, gid):
    return os.path.join(SED_ROOT, f'snap_{int(snap):03d}', f'gal_{int(gid)}',
                        f'snap{int(snap):03d}.galaxy{int(gid):06d}.rtout.sed')

POOL['HAS_SED'] = np.array([os.path.exists(rtout_path(ANCHOR['snap'], g)) for g in POOL['gal_id']])
print(f"anchor pool (snap {ANCHOR['snap']}, z={ANCHOR['z_snap']}): {len(POOL)} quenched galaxies, "
      f"{int(POOL['HAS_SED'].sum())} with completed RT, {int(POOL['IS_AGN'].sum())} AGN hosts")
if POOL['HAS_SED'].sum() == 0:
    print('\n*** no RT output yet -> submit the staged job:\n'
          f'    sbatch {os.path.join(SED_ROOT, "snap_105", "master.snap105.job")}\n'
          '    then re-run from Part 4 ***')


In [ ]:
# ── Part 3 · analogue selection + sanity (mass/age vs observed) ──────────────
ANALOGUES = {}
for tid, T in TARGETS.items():
    m_mass = np.abs(np.asarray(POOL['LOG_MSTAR']) - T['logm']) <= T['dlogm']
    m_age  = np.abs(np.asarray(POOL['AGE_STAR']) - T['age_gyr']) <= T['dage_gyr']
    sel = m_mass & m_age
    agn = np.asarray(POOL['IS_AGN'])
    print(f"── {tid} ──  |dlogM*|<={T['dlogm']}: {int(m_mass.sum())}"
          f"  & |dage|<={T['dage_gyr']}: {int(sel.sum())}"
          f"  -> AGN {int((sel & agn).sum())} | non-AGN {int((sel & ~agn).sum())}"
          f"  (with SED: {int((sel & POOL['HAS_SED']).sum())})")
    A = POOL[sel]
    A['AGN_CLASS'] = np.where(np.asarray(A['IS_AGN']), 'AGN', 'no_AGN')
    A.sort('gal_id')
    out = os.path.join(OUTDIR, f'analogues_cis25_{tid}.fits')
    A.write(out, overwrite=True)
    ANALOGUES[tid] = A
    print(f"  -> {out}")
    A['gal_id', 'LOG_MSTAR', 'AGE_STAR', 'ssfr', 'LOG_MDUST', 'FEDD', 'AGN_CLASS',
      'agn_class'].pprint(max_lines=-1, max_width=-1)   # agn_class = m25 feedback-mode tercile label
    print()

OBS_RANGES = {'719':  dict(logm=(10.49, 10.59), age=(2.1, 3.0)),
              '2962': dict(logm=(10.74, 10.97), age=(3.1, 3.6))}
fig, axes = plt.subplots(len(TARGETS), 2, figsize=(10, 4 * len(TARGETS)))
for irow, (tid, T) in enumerate(TARGETS.items()):
    A, R = ANALOGUES[tid], OBS_RANGES[tid]
    for icol, (key, xlabel, obs_val, obs_rng) in enumerate([
            ('LOG_MSTAR', r'$\log\,M_*\,[M_\odot]$', T['logm'],    R['logm']),
            ('AGE_STAR',  'mass-weighted age [Gyr]', T['age_gyr'], R['age'])]):
        ax = axes[irow, icol]
        xall = np.asarray(POOL[key], float)
        x = np.asarray(A[key], float)
        bins = np.linspace(np.nanmin(xall) - 0.1, max(np.nanmax(xall), obs_val) + 0.1, 22)
        ax.hist(xall, bins=bins, color='0.9', label='anchor pool (44)')
        ax.hist(x, bins=bins, color=COL['no_AGN'], alpha=0.75, label='selected analogues')
        ax.axvspan(obs_rng[0], obs_rng[1], color='k', alpha=0.12, lw=0, label='obs. consensus')
        ax.axvline(obs_val, color='k', lw=1.6, label='observed')
        ax.axvline(float(np.median(x)), color='crimson', lw=1.4, ls='--', label='analogue median')
        ax.set_xlabel(xlabel)
        if icol == 0:
            ax.set_ylabel(f'nirspec7-{tid.zfill(6)}\nN')
        if irow == 0 and icol == 1:
            ax.legend(fontsize=9, frameon=False)
        print(f'{tid} {key:9s}: obs {obs_val:6.2f} | analogue median {float(np.median(x)):6.2f}')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'analogue_property_sanity_cis25.{ext}'), dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ── Part 4 · powderday SED stacks vs observed CIGALE SED ─────────────────────
# m25 runs store 4 sightlines (THETA 0/45/90/135) and 5 apertures; each sightline enters the
# stack as its own curve (viewing-angle scatter), aperture fixed by APERTURE_IDX.
from hyperion.model import ModelOutput

MAX_INDIV = 40
RESCALE_TO_TARGET_MASS = True     # multiply each SED by M*_target / M*_analogue

_SED_CACHE = {}
def read_powderday_sed(snap, gid):
    """(wav_rest_um, nuLnu_erg_s [n_incl, n_wav]) at APERTURE_IDX; None if RT output missing."""
    key = (int(snap), int(gid))
    if key not in _SED_CACHE:
        p = rtout_path(snap, gid)
        if not os.path.exists(p):
            _SED_CACHE[key] = None
        else:
            m = ModelOutput(p)
            sed = m.get_sed(inclination='all', aperture=APERTURE_IDX)
            wav = np.asarray(sed.wav, float)                      # micron, rest frame
            val = np.atleast_2d(np.asarray(sed.val, float))      # (n_incl, n_wav), nu*Lnu [erg/s]
            s = np.argsort(wav)
            _SED_CACHE[key] = (wav[s], val[:, s])
    return _SED_CACHE[key]

GRID = np.geomspace(0.3, 1000.0, 300)     # observed-frame micron

def stack(A_group, z_target, tag='', logm_target=None):
    """Median + 16/84 Fnu(GRID); every (galaxy, sightline) is one curve. None if no data."""
    dl = Planck13.luminosity_distance(z_target).to(u.cm)
    curves, n_missing = [], 0
    for i, r in enumerate(A_group):
        out = read_powderday_sed(ANCHOR['snap'], r['gal_id'])
        if out is None:
            n_missing += 1
            continue
        wav_rest, nuLnu = out
        wl_obs = wav_rest * (1.0 + z_target)
        nu = (const.c / (wl_obs * u.micron)).to(u.Hz).value
        scale = 10.0 ** (logm_target - float(r['LOG_MSTAR'])) if logm_target is not None else 1.0
        for v in nuLnu:                                          # one curve per sightline
            fnu = (v * u.erg / u.s / (4 * np.pi * dl**2)).to(u.mJy * u.Hz).value / nu * scale
            with np.errstate(invalid='ignore', divide='ignore'):
                lg = np.interp(np.log10(GRID), np.log10(wl_obs), np.log10(fnu),
                               left=np.nan, right=np.nan)
            curves.append(10**lg)
    if n_missing:
        print(f'  [{tag}] {n_missing}/{len(A_group)} galaxies have no RT output yet (skipped)')
    if not curves:
        return None
    C = np.asarray(curves)
    return dict(med=np.nanmedian(C, axis=0), lo=np.nanpercentile(C, 16, axis=0),
                hi=np.nanpercentile(C, 84, axis=0), curves=C)

rng_sed = np.random.default_rng(7)
for tid, T in TARGETS.items():
    A = ANALOGUES[tid]
    _logm_t = T['logm'] if RESCALE_TO_TARGET_MASS else None
    fig, ax = plt.subplots(figsize=(9, 6.5))
    for cls in ('no_AGN', 'AGN'):
        G = A[np.asarray(A['AGN_CLASS']) == cls]
        if len(G) == 0:
            continue
        S = stack(G, T['z'], tag=f'{tid}/{cls}', logm_target=_logm_t)
        if S is None:
            continue
        c = COL[cls]
        idx = np.arange(len(S['curves']))
        if idx.size > MAX_INDIV:
            idx = rng_sed.choice(idx, MAX_INDIV, replace=False)
        for cv in S['curves'][idx]:
            ax.plot(GRID, cv, color=c, lw=0.6, alpha=0.15)
        ax.fill_between(GRID, S['lo'], S['hi'], color=c, alpha=0.28, lw=0)
        lab = 'AGN host' if cls == 'AGN' else 'non-AGN host'
        ax.plot(GRID, S['med'], color=c, lw=2,
                label=f'SIMBA-25 {lab} (N={len(G)} gal x 4 los)')

    O = OBS[tid]
    ax.plot(O['model_wl'], O['model_tot'], color=COL['obs'], lw=1.6,
            label='CIGALE best model (total)')
    if T['agn']:
        ax.plot(O['model_wl'], O['model_gal'], color=COL['obs'], lw=1.4, ls='--',
                label='CIGALE galaxy only (AGN subtracted)')
    ax.errorbar(O['phot_wl'], O['phot_f'], yerr=np.abs(O['phot_e']), fmt='o', ms=6,
                mfc='w', mec=COL['obs'], ecolor=COL['obs'], elinewidth=1, capsize=2,
                zorder=5, label='observed photometry')

    ax.set(xscale='log', yscale='log', xlim=(0.3, 1000),
           xlabel=r'observed wavelength [$\mu$m]', ylabel=r'$F_\nu$ [mJy]')
    fin = O['phot_f'][np.isfinite(O['phot_f'])]
    ax.set_ylim(max(1e-6, fin.min() / 3e2), fin.max() * 3e2)
    ax.grid(alpha=0.2, which='both', lw=0.5)
    ax.legend(loc='lower center', fontsize=10, frameon=False)
    _rs = ', mass-rescaled' if RESCALE_TO_TARGET_MASS else ''
    ax.set_title(f'nirspec7-{tid.zfill(6)}  (z={T["z"]})  vs  SIMBA-25 analogues '
                 f'(snap 105, placed at z={T["z"]}{_rs})', fontsize=12)
    fig.tight_layout()
    _suffix = '_massrescaled' if RESCALE_TO_TARGET_MASS else ''
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(OUTDIR, f'sed_analogues_cis25_{tid}{_suffix}.{ext}'),
                    dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Part 5 · dust masses: simulation prediction vs CIGALE ────────────────────
rng = np.random.default_rng(4)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
summary = []
for ax, (tid, T) in zip(axes, TARGETS.items()):
    A = ANALOGUES[tid]
    for x0, cls in ((0.0, 'no_AGN'), (1.0, 'AGN')):
        G = A[np.asarray(A['AGN_CLASS']) == cls]
        y = np.asarray(G['LOG_MDUST'], float)
        y = y[np.isfinite(y)]
        if y.size == 0:
            continue
        x = x0 + rng.uniform(-0.14, 0.14, y.size)
        ax.plot(x, y, 'o', ms=5.5, color=COL[cls], alpha=0.6, mec='none')
        med, lo, hi = np.median(y), np.percentile(y, 16), np.percentile(y, 84)
        ax.hlines(med, x0 - 0.24, x0 + 0.24, color=COL[cls], lw=2.5)
        ax.vlines(x0 - 0.24, lo, hi, color=COL[cls], lw=1.2)
        summary.append(dict(target=tid, group=cls, N=int(y.size), med_logMd=round(float(med), 2),
                            p16=round(float(lo), 2), p84=round(float(hi), 2)))
    if tid == '719':
        ax.errorbar(0.5, T['logmdust'], yerr=[[0.0], [0.0]], uplims=True, fmt='s', ms=9,
                    color=COL['obs'], capsize=4, label='CIGALE (~upper limit)')
    else:
        ax.errorbar(0.5, T['logmdust'], yerr=T['logmdust_err'], fmt='s', ms=9,
                    color=COL['obs'], capsize=4, label='CIGALE')
    ax.set(xticks=[0, 0.5, 1], xticklabels=['non-AGN\nhosts', 'observed', 'AGN\nhosts'],
           xlim=(-0.5, 1.5), title=f'nirspec7-{tid.zfill(6)}')
    ax.grid(alpha=0.2, axis='y', lw=0.5)
axes[0].set_ylabel(r'$\log\,M_{\rm dust}\,[M_\odot]$')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUTDIR, f'dust_mass_analogues_cis25.{ext}'), dpi=200, bbox_inches='tight')
plt.show()

SUM = Table(summary)
SUM.write(os.path.join(OUTDIR, 'dust_mass_summary_cis25.fits'), overwrite=True)
SUM.pprint(max_width=-1)
print(f"\nobserved: 719 log Md ~ {TARGETS['719']['logmdust']} (upper-limit-like), "
      f"2962 log Md = {TARGETS['2962']['logmdust']} +/- {TARGETS['2962']['logmdust_err']}")


## Part 6 · status & caveats

**To get the SEDs**: the snap 105 dust-on RT is fully staged but not run —

```
cd output/cis25/sed_quenched_regions/dusty_simdust/powderday_sed_out/snap_105
sbatch master.snap105.job
```

then re-run Part 2 onward (Part 4 skips galaxies whose `.rtout.sed` is still missing and says so).

**Caveats**
- All analogues sit at the z = 1.0 anchor (z = 0.993): Δt ≈ 0.6 Gyr (2962) / 1.1 Gyr (719) after
  the observed epochs. They are placed at the target redshift for flux normalization.
- No AGN-host analogues exist in the 25 Mpc box at this anchor — the AGN/non-AGN split is
  non-AGN-only. `agn_class` (the m25 feedback-mode tercile over the quench window) is carried in
  the selection tables as an alternative, feedback-based label.
- The anchor pool's dust masses are low (median log Md ≈ 5) — expect the FIR comparison, not the
  optical, to show the differences (that is the point of the exercise).
- Cosmology Planck13 for fluxes (as in `makesed`); ages from `history_anchor_z1.hdf5` row 0.
